# QML Fraud Detection Benchmark — Noise Sweep Results

> **Client:** Leading Financial Service Provider  
> **Question:** At what depolarizing noise level do quantum models fall below classical baselines?

This notebook presents the results of a depolarizing noise sweep comparing:
- **VQC** (Variational Quantum Classifier) — StronglyEntanglingLayers, 8 qubits, 30 epochs
- **QSVM** (Quantum Support Vector Machine) — quantum kernel, 8 qubits
- **Random Forest** and **XGBoost** — classical baselines (noise-independent)

Noise levels swept: `p = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]`  
Current best NISQ hardware: p ≈ 0.001

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load Results

In [ ]:
with open('../results/noise/noise_results.json') as f:
    results = json.load(f)

df = pd.DataFrame([
    {
        'model': r['model'],
        'noise_level': r['noise_level'],
        **r['metrics']
    }
    for r in results
])

classical = df[df['noise_level'].isna()].copy()
quantum   = df[df['noise_level'].notna()].copy()

print(f"Records: {len(df)} ({len(classical)} classical, {len(quantum)} quantum)")
df.head(10)

## 2. Summary Table

In [ ]:
metrics = ['f1_fraud', 'mcc', 'pr_auc', 'roc_auc']

# Classical baselines
print("Classical Baselines (noise-independent)")
print(classical[['model'] + metrics].to_string(index=False))

print("\nQuantum Models by Noise Level")
pivot = quantum.pivot_table(index='noise_level', columns='model', values='f1_fraud')
print(pivot.round(3).to_string())

## 3. Noise Sweep Plot

In [ ]:
from IPython.display import Image
Image('../results/noise/noise_vs_metric.png')

## 4. Key Finding: At What Noise Level Does Quantum Break Down?

In [ ]:
xgb_f1 = classical[classical['model'] == 'XGBoost']['f1_fraud'].values[0]
rf_f1  = classical[classical['model'] == 'Random Forest']['f1_fraud'].values[0]

fig, ax = plt.subplots(figsize=(9, 5))

colors = {'VQC': '#D65F5F', 'QSVM': '#B47CC7'}
for model, grp in quantum.groupby('model'):
    grp = grp.sort_values('noise_level')
    ax.plot(grp['noise_level'], grp['f1_fraud'],
            marker='o', linewidth=2, label=model, color=colors.get(model, 'gray'))

ax.axhline(xgb_f1, linestyle='--', color='#6ACC65', linewidth=1.5, label=f'XGBoost (F1={xgb_f1:.3f})')
ax.axhline(rf_f1,  linestyle='--', color='#4878CF', linewidth=1.5, label=f'Random Forest (F1={rf_f1:.3f})')
ax.axvspan(0.001, 0.01, alpha=0.08, color='orange', label='Current NISQ range')

ax.set_xlabel('Depolarizing error probability p')
ax.set_ylabel('F1-Fraud')
ax.set_title('Quantum vs Classical: F1-Fraud under Depolarizing Noise')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 5. Conclusions

<!-- TODO: fill in after all results are collected -->

- **QSVM robustness:** ...
- **VQC limitations:** ...
- **Classical ceiling:** ...
- **NISQ viability verdict:** ...